In [ ]:
import polars as pl

df = pl.read_parquet("../../data/clean/flights_clean.parquet")
print(df.shape)

(2920860, 33)


In [3]:
df = df.with_columns([
    pl.col("FL_DATE").str.to_date("%Y-%m-%d").alias("date"),
]).with_columns([
    pl.col("date").dt.weekday().alias("day_of_week"),
    pl.col("date").dt.month().alias("month"),
    pl.col("date").dt.year().alias("year"),
    (pl.col("date").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
])

print(df.columns)

['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'COVID_FLAG', 'date', 'day_of_week', 'month', 'year', 'is_weekend']


In [4]:
df = df.with_columns([
    (pl.col("CRS_ARR_TIME") // 100).alias("arr_hour"),
    ((pl.col("CRS_ARR_TIME") % 100) // 15).alias("arr_15min_block")
])

In [5]:
throughput = df.group_by(["DEST", "date", "arr_hour", "arr_15min_block"]).agg(
    pl.len().alias("arrivals"),
    pl.col("day_of_week").first(),
    pl.col("month").first(),
    pl.col("year").first(),
    pl.col("is_weekend").first(),
    pl.col("COVID_FLAG").first()
).sort(["DEST", "date", "arr_hour", "arr_15min_block"])

print(throughput.shape)
throughput.head(10)

(2288702, 10)


DEST,date,arr_hour,arr_15min_block,arrivals,day_of_week,month,year,is_weekend,COVID_FLAG
str,date,i64,i64,u32,i8,i8,i32,i8,i32
"""ABE""",2019-01-02,12,0,1,3,1,2019,0,0
"""ABE""",2019-01-02,17,1,1,3,1,2019,0,0
"""ABE""",2019-01-02,22,1,1,3,1,2019,0,0
"""ABE""",2019-01-04,12,0,1,5,1,2019,1,0
"""ABE""",2019-01-04,22,1,1,5,1,2019,1,0
"""ABE""",2019-01-05,10,3,1,6,1,2019,1,0
"""ABE""",2019-01-06,17,1,1,7,1,2019,1,0
"""ABE""",2019-01-07,16,1,1,1,1,2019,0,0
"""ABE""",2019-01-07,17,1,1,1,1,2019,0,0


In [6]:
throughput = throughput.with_columns([
    pl.col("arrivals")
    .mean()
    .over(["DEST", "arr_hour", "day_of_week"])
    .alias("hist_mean_arrivals")
])

print(throughput.select(["DEST", "arr_hour", "day_of_week", 
                          "arrivals", "hist_mean_arrivals"]).head(10))

shape: (10, 5)
┌──────┬──────────┬─────────────┬──────────┬────────────────────┐
│ DEST ┆ arr_hour ┆ day_of_week ┆ arrivals ┆ hist_mean_arrivals │
│ ---  ┆ ---      ┆ ---         ┆ ---      ┆ ---                │
│ str  ┆ i64      ┆ i8          ┆ u32      ┆ f64                │
╞══════╪══════════╪═════════════╪══════════╪════════════════════╡
│ ABE  ┆ 12       ┆ 3           ┆ 1        ┆ 1.0                │
│ ABE  ┆ 17       ┆ 3           ┆ 1        ┆ 1.071429           │
│ ABE  ┆ 22       ┆ 3           ┆ 1        ┆ 1.0                │
│ ABE  ┆ 12       ┆ 5           ┆ 1        ┆ 1.0                │
│ ABE  ┆ 22       ┆ 5           ┆ 1        ┆ 1.06383            │
│ ABE  ┆ 10       ┆ 6           ┆ 1        ┆ 1.0                │
│ ABE  ┆ 17       ┆ 7           ┆ 1        ┆ 1.0                │
│ ABE  ┆ 16       ┆ 1           ┆ 1        ┆ 1.064516           │
│ ABE  ┆ 17       ┆ 1           ┆ 1        ┆ 1.0                │
│ ABE  ┆ 16       ┆ 2           ┆ 1        ┆ 1.0             

In [7]:
import os
os.makedirs("../data/clean", exist_ok=True)
throughput.write_parquet("../data/clean/airport_throughput.parquet")
print("saved:", throughput.shape)

saved: (2288702, 11)


In [8]:
throughput.filter(
    (pl.col("DEST") == "ATL") & 
    (pl.col("day_of_week") == 1) &
    (pl.col("COVID_FLAG") == 0)
).sort("arr_hour").head(20)

DEST,date,arr_hour,arr_15min_block,arrivals,day_of_week,month,year,is_weekend,COVID_FLAG,hist_mean_arrivals
str,date,i64,i64,u32,i8,i8,i32,i8,i32,f64
"""ATL""",2019-01-07,0,2,1,1,1,2019,0,0,1.106667
"""ATL""",2019-01-21,0,3,1,1,1,2019,0,0,1.106667
"""ATL""",2019-01-28,0,1,1,1,1,2019,0,0,1.106667
"""ATL""",2019-02-04,0,2,2,1,2,2019,0,0,1.106667
"""ATL""",2019-02-11,0,0,1,1,2,2019,0,0,1.106667
…,…,…,…,…,…,…,…,…,…,…
"""ATL""",2019-06-10,0,0,1,1,6,2019,0,0,1.106667
"""ATL""",2019-06-10,0,2,1,1,6,2019,0,0,1.106667
"""ATL""",2019-06-24,0,1,1,1,6,2019,0,0,1.106667
